In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/data.csv")

In [2]:
df.head()

,game_id,subframe,won,playfield,x,y,r,placed,hold,next,...,garbage_cleared,attack,t_spin,btb,combo,immediate_garbage,incoming_garbage,rating,glicko,glicko_rd
0,1,67,1,NaN,4,0,N,I,N,JZSOTLSLIOJZTJ,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
1,1,170,1,NNNIIII,4,1,N,Z,J,SOTLSLIOJZTJTZ,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
2,1,339,1,NNNIIIINNNNNNNZZNNNNNNNZZ,6,1,E,S,J,OTLSLIOJZTJTZS,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
3,1,472,1,NNNIIIISNNNNNNZZSSNNNNNZZNS,8,0,N,O,J,TLSLIOJZTJTZSO,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
4,1,602,1,NNNIIIISOONNNNZZSSOONNNZZNS,0,1,E,J,T,LSLIOJZTJTZSOI,...,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624


### Removing Unnecessary Variables

In [2]:

unneeded_cols = ["playfield", "x", "y", "r", "next", "hold", "placed"]
present_unneeded = [col for col in unneeded_cols if col in df.columns]
df = df.drop(columns=present_unneeded)

In [4]:
df.head()

,game_id,subframe,won,cleared,garbage_cleared,attack,t_spin,btb,combo,immediate_garbage,incoming_garbage,rating,glicko,glicko_rd
0,1,67,1,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
1,1,170,1,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
2,1,339,1,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
3,1,472,1,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
4,1,602,1,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624


### Checking for Multiple Representations

In [3]:
df['won'].value_counts()

won
1    4274700
0    3441824
Name: count, dtype: int64

In [6]:
df['t_spin'].value_counts()

t_spin
N    7150022
F     527604
M      38898
Name: count, dtype: int64

### Checking For Missing Data

In [14]:
df.isnull().any()

game_id              False
subframe             False
won                  False
cleared              False
garbage_cleared      False
attack               False
t_spin               False
btb                  False
combo                False
immediate_garbage    False
incoming_garbage     False
rating               False
glicko               False
glicko_rd            False
dtype: bool

### Checking for Negative Values

In [21]:
numeric_cols = df.select_dtypes(include=['number'])
negative_counts = (numeric_cols < 0).sum()
print(negative_counts)

game_id              0
subframe             0
won                  0
cleared              0
garbage_cleared      0
attack               0
btb                  0
combo                0
immediate_garbage    0
incoming_garbage     0
rating               0
glicko               0
glicko_rd            0
dtype: int64


### Checking for Incorrect Datatypes

In [4]:
df.dtypes

game_id                int64
subframe               int64
won                    int64
cleared                int64
garbage_cleared        int64
attack                 int64
t_spin                object
btb                    int64
combo                  int64
immediate_garbage      int64
incoming_garbage       int64
rating               float64
glicko               float64
glicko_rd            float64
dtype: object

In [ ]:
# Convert integer columns to smallest possible integer type,
# and float columns to smallest possible float type.
int_cols = [
    "won",
    "game_id",
    "subframe",
    "cleared",
    "garbage_cleared",
    "attack",
    "btb",
    "combo",
    "immediate_garbage",
    "incoming_garbage",
]
float_cols = ["rating", "glicko", "glicko_rd"]
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast="integer", errors="coerce")
for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast="float", errors="coerce")

# state that this is for optimization purposes

In [ ]:
df.dtypes

### Checking for Default Values and Inconsistent Formatting

In [5]:
for col in df:
  print(f"{col}: {df[col].unique()}")
  print("")

game_id: [     1      2      3 ... 124030 124031 124032]

subframe: [    67    170    339 ... 102556 103224 103661]

won: [1 0]

cleared: [0 2 1 4 3]

garbage_cleared: [0]

attack: [ 0  4  1  5  3  6  2  7  8 10  9 11 12 14 15 16 17 13 19 18 22 20]

t_spin: ['N' 'F' 'M']

btb: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54]

combo: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]

immediate_garbage: [ 0  7  8  4  2  1  6 10 13  5  3  9 11 15 12 14 20 16 19 23 22 18 28 17
 21 26 25 24 31 27 32 36 30 40 35 41 33 29 34 38 37 69 78 49 42 50 43 51
 39 46 59 58 77 61 52 56]

incoming_garbage: [ 0  7 15  9  2  1  4  6  5  3 13 10 11  8 16 12 14 18 20 19 23 29 17 28
 22 24 21 26 25 27 31 32 34 36 30 40 35 41 33 38 37 69 78 49 42 50 43 51
 39 46 66 58 77 74 52 56 59 44]

rating: [24748.52148438 24820.7109375  24788.74414062 24830.14648438
 24859.94726562 24

### Checking for Duplicate Data

In [20]:
df.duplicated().any()

np.True_

In [16]:
df = df.drop_duplicates(subset=['game_id', 'subframe'])
# only drop the duplicated where it has the same game_id and subframe
# it is physically and mechanically impossible for a player to place two different pieces at the exact same subframe. A player can only drop one piece at an exact millisecond
df.shape

(7716512, 14)

### Checking for Inconsistent Games

In [ ]:
# Drop games where 'won', 'rating', 'glicko', or 'glicko_rd' are not consistent within a game.
if {"game_id", "won", "rating", "glicko", "glicko_rd"}.issubset(df.columns):
    gchecks = df.groupby("game_id")[
        ["won", "rating", "glicko", "glicko_rd"]].nunique()
    inconsistent_games = gchecks[(gchecks > 1).any(axis=1)].index.tolist()
    print(f"Inconsistent games found: {len(inconsistent_games)}")
    df = df[~df["game_id"].isin(inconsistent_games)]

Inconsistent games found: 0


### Checking for Outliers

In [12]:
# Print min, max, and extreme quantiles for key numeric columns.
num_cols = [
    "cleared",
    "garbage_cleared",
    "attack",
    "btb",
    "combo",
    "immediate_garbage",
    "incoming_garbage",
]
for col in num_cols:
    if col in df.columns:
        high = df[col].quantile(0.999)
        low = df[col].quantile(0.001)
        print(f"{col}: min={df[col].min()}, max={df[col].max()}, 0.1%={
            low}, 99.9%={high}")

cleared: min=0, max=4, 0.1%=0.0, 99.9%=4.0
garbage_cleared: min=0, max=0, 0.1%=0.0, 99.9%=0.0
attack: min=0, max=22, 0.1%=0.0, 99.9%=7.0
btb: min=0, max=54, 0.1%=0.0, 99.9%=19.0
combo: min=0, max=21, 0.1%=0.0, 99.9%=7.0
immediate_garbage: min=0, max=78, 0.1%=0.0, 99.9%=15.0
incoming_garbage: min=0, max=78, 0.1%=0.0, 99.9%=17.0


In [ ]:
num_cols = ["cleared", "garbage_cleared", "attack", "btb", "combo", "immediate_garbage", "incoming_garbage"]

for col in num_cols:
    if col in df.columns:
        top_values = df[col].value_counts().nlargest(5).to_dict()
        
        max_val = df[col].max()
        
        print(f"--- {col} ---")
        print(f"Max Value: {max_val}")
        print(f"Top 5 most common values: {top_values}\n")

--- cleared ---
Max Value: 4
Top 5 most common values: {0: 5587295, 1: 958295, 2: 690330, 4: 381767, 3: 98837}

--- garbage_cleared ---
Max Value: 0
Top 5 most common values: {0: 7716524}

--- attack ---
Max Value: 22
Top 5 most common values: {0: 6033922, 1: 590235, 4: 439143, 2: 207244, 5: 197437}

--- btb ---
Max Value: 54
Top 5 most common values: {0: 3580661, 1: 1467989, 2: 890309, 3: 553606, 4: 388236}

--- combo ---
Max Value: 21
Top 5 most common values: {0: 5596521, 1: 1315814, 2: 445764, 3: 197292, 4: 88032}

--- immediate_garbage ---
Max Value: 78
Top 5 most common values: {0: 6690420, 1: 243668, 5: 150719, 4: 149941, 2: 148039}

--- incoming_garbage ---
Max Value: 78
Top 5 most common values: {0: 5866757, 1: 380037, 5: 290027, 4: 277661, 2: 265844}



### Feature Engineering

In [ ]:
print("CLEANING SUMMARY:")
print(f"Rows after cleaning: {len(df)}")
print(f"Unique games: {df['game_id'].nunique()
      if 'game_id' in df.columns else 'N/A'}")

print("Num placement rows:", len(df))
print("Num unique game_id:", df["game_id"].nunique())
print("game_id dtype:", df["game_id"].dtype)

# Aggregation
# groupby game_id
agg_dict = {
    "subframe": ["max", "count"],
    "cleared": "sum",
    "garbage_cleared": "sum",
    "attack": "sum",
    "t_spin": lambda x: x.isin(["M", "F"]).sum(), 
    "btb": ["mean", "max"],
    "combo": ["mean", "max"],
    "immediate_garbage": ["mean", "max"],
    "incoming_garbage": ["mean", "max"],
    "won": "first",
    "rating": "first",
    "glicko": "first",
    "glicko_rd": "first",
}

game_level = df.groupby("game_id").agg(agg_dict)

game_level.columns = ["_".join([c for c in col if c]) for col in game_level.columns]
game_level = game_level.reset_index()

rename_map = {
    "t_spin_<lambda>": "t_spin_count",
    "won_first": "won",
    "rating_first": "rating",
    "glicko_first": "glicko",
    "glicko_rd_first": "glicko_rd"
}
game_level = game_level.rename(columns=rename_map)

game_level["duration_sec"] = game_level["subframe_max"] / 600
game_level["pps"] = game_level["subframe_count"] / game_level["duration_sec"]
game_level["attack_per_piece"] = game_level["attack_sum"] / game_level["subframe_count"]
game_level["apm"] = (game_level["attack_sum"] / game_level["duration_sec"]) * 60
game_level["tspin_rate"] = game_level["t_spin_count"] / game_level["subframe_count"]

print("Num rows in game_level:", len(game_level))

for col in game_level:
  print(f"{col}: {game_level[col].unique()}")
  print("")

# Export
# game_level.to_csv("./data/game_level.csv", index=False)

CLEANING SUMMARY:
Rows after cleaning: 7716512
Unique games: 76692
Num placement rows: 7716512
Num unique game_id: 76692
game_id dtype: int64
Num rows in game_level: 76692
game_id: [     1      2      3 ... 124030 124031 124032]

subframe_max: [42360 42786 29565 ... 20760 19859 41873]

subframe_count: [196 215 152 136 191 197 123 138  28  33  32  42  40 139 129  56  50  35
  64  34  11  13 333 134 300 119  22  70  31  74  53   5 179 193 374 380
  94  76  68  75 398 410 106 108  63  48  26  15  84 111  39  25  44  52
  47 159 166 103  69  72  30  49  82 248  87  67  61  45  86  73  27 228
 264  77  85  79  92 101 115 107  62  59 145  97  54 171 120  83  80  23
 163 122 279 320 116 102  95 231 223 156  29  99 150 131  17 204 125 105
  93 205 199  38 275  14 209 175 406  43 154 184 172 113  46  65  24 100
  58  60  89  36  91 165 155  20 192  51 135 151 128 190 189  19 140 142
 141 137  21 250  37 176 114  71  81  78 132 130 425 182 164 153 146 149
 104 201 493 162 356 350 238 270  55  57

### Further Preprocessing

In [37]:
# drop games that lasted for 10 seconds or less to rule out instant disconnects/forfeits
game_level2 = game_level[game_level["duration_sec"] >= 10.0].copy()
game_level2.shape

(70246, 24)

In [39]:
# drop intermediate columns and identifier
cols_to_drop = [
    "game_id",
    "subframe_max",
    "subframe_count",
    "cleared_sum",
    "garbage_cleared_sum",   
    "attack_sum",
    "t_spin_count"
]

final_df = game_level.drop(columns=cols_to_drop)
final_df.head()

,btb_mean,btb_max,combo_mean,combo_max,immediate_garbage_mean,immediate_garbage_max,incoming_garbage_mean,incoming_garbage_max,won,rating,glicko,glicko_rd,duration_sec,pps,attack_per_piece,apm,tspin_rate
0,1.933673,8,0.520408,7,0.244898,8,0.576531,15,1,24748.521484,2701.887695,62.212624,70.600000,2.776204,0.821429,136.827195,0.076531
1,0.413953,4,0.590698,6,0.683721,13,1.432558,13,0,24820.710938,2791.866943,63.886936,71.310000,3.015005,0.586047,106.015987,0.055814
2,0.611842,3,0.605263,8,0.250000,6,0.605263,6,1,24820.710938,2791.866943,63.886936,49.275000,3.084729,0.671053,124.200913,0.065789
3,0.786765,4,0.588235,6,0.551471,9,1.227941,10,0,24748.521484,2701.887695,62.212624,49.283333,2.759554,0.625000,103.483260,0.036765
4,1.277487,7,0.539267,5,0.366492,11,0.691099,11,1,24748.521484,2701.887695,62.212624,69.525000,2.747213,0.670157,110.463862,0.057592
